# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and analyze the FAIR² dataset using the `mlcroissant` library. The dataset describes proteins identified via mass spectrometry analysis of human extracellular vesicles, including abundance, modifications, and associated parameters.

### Dataset Source

The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s. All entities are referenced exclusively by their `@id` for clarity and reproducibility.

Let's print the available record sets, and for one, also print its available fields (columns and their `@id`).

In [ ]:
# List available record sets and their fields/columns by @id

record_set_ids = [rs['@id'] for rs in getattr(metadata, 'recordSet', [])]
if not record_set_ids:
    print("No record sets defined in top-level metadata. Attempting to auto-discover from Croissant schema...")
    # fallback: iterate through files and try to infer tables (for Croissant datasets with minimal markup)
    # This is for demonstration; ideally, recordSets should be defined with @id
    try:
        # Try to load the record sets dynamically (mlcroissant>0.5.0 allows listing them)
        record_sets = dataset.list_record_sets()
        record_set_ids = [rs['@id'] for rs in record_sets]
    except Exception as e:
        print("Could not auto-discover record sets.")
        record_set_ids = []

print("Available record sets in this dataset (by @id):")
for rsid in record_set_ids:
    print(f"- {rsid}")

# For demonstration, select the first record set @id
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nFields in record set '{main_record_set_id}':")
    try:
        # Print columns (fields) of the first record set
        columns = dataset.list_fields(record_set=main_record_set_id)
        for field in columns:
            print(f"  {field['@id']}: {field.get('name')}")
    except Exception:
        print("No columns/fields could be listed for this record set.")
else:
    print("No record sets discovered in this dataset.")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from the discovered record sets

dataframes = {}

if not record_set_ids:
    print("No record sets to extract.")
else:
    for rsid in record_set_ids:
        try:
            records = list(dataset.records(record_set=rsid))
            df = pd.DataFrame(records)
            dataframes[rsid] = df
            print(f"Loaded {len(df)} records from {rsid}.")
        except Exception as e:
            print(f"Could not load records from {rsid}: {e}")

    # Display columns for the main record set
    print("\nColumns for the first record set:")
    main_df = dataframes[main_record_set_id]
    print(main_df.columns.tolist())
    main_df.head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations may include removing outliers, transforming distributions, or grouping by key attributes to prepare data for analysis.

In [ ]:
# Select a numeric field and a group field for demonstration

df = dataframes.get(main_record_set_id)
if df is not None:
    # Guess a numeric field (commonly mass, abundance, or count)
    # Find a numeric column (float/int) from the DataFrame's dtypes
    numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]
        print(f"Using numeric field: {numeric_field_id}")
    else:
        print("No numeric fields found.")
        numeric_field_id = None

    # Filter records where numeric_field > threshold
    threshold = 10
    if numeric_field_id:
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the selected numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by another string/categorical field
        categorical_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
        group_field = categorical_fields[0] if categorical_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"\nGrouped data by {group_field} (mean of {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No suitable numeric field for EDA.")
else:
    print("No data loaded for EDA.")

## 5. Visualization

Visualize distributions or relationships between fields in the dataset.

Below, we show a histogram (if a numeric field was found) and a boxplot by group if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No appropriate fields for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to:

- Load FAIR² protein mass spectrometry data using the `mlcroissant` library via its Croissant schema URL.
- Discover tabular data and view available fields (referencing all by their `@id`).
- Extract data into DataFrames and perform filtering, normalization, and grouping.
- Visualize data distributions with basic histograms and boxplots.

Further analysis could include more detailed protein clustering, outlier analysis, or downstream bioinformatics depending on your research questions.

_All field and record set references in this notebook used their canonical Croissant `@id`._